## GAWP Sandbox

Experiments with reading data files for germline analysis

In [1]:
import os
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_DIR = Path.home() / 'Research/Projects/LibudaLab/GAP/PRG-1'
DATA_DIR = PROJECT_DIR / 'xlsx'
OUTPUT_DIR = PROJECT_DIR / 'Demo'
MEASUREMENTS = PROJECT_DIR / 'Germline Measurements.xlsx'


In [3]:
DATA_DIR

PosixPath('/Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/xlsx')

#### Data Format

The data directory has three spreadsheets:

```shell
xlsx
├── 210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx
├── 210924_DLW67_noAUX_HS_RAD51_AID_herm_g07_stitchedtif.xlsx
└── 210924_DLW67_noAUX_HS_RAD51_AID_herm_g09_stitched.xlsx
```

When viewed in Excel there are 39 tabs:
* Overall describes the data
* Intensity max, mean, and sum for three channels (9 tabs in all)
* Position X, Y, and Z and combined
* Sphericity, Volume, distances



### Read Data File

Get a list of file names in the data directory.

In [4]:
data_files = sorted(os.listdir(DATA_DIR))

In [5]:
data_files

['210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx',
 '210924_DLW67_noAUX_HS_RAD51_AID_herm_g07_stitchedtif.xlsx',
 '210924_DLW67_noAUX_HS_RAD51_AID_herm_g09_stitched.xlsx',
 '~$210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx']

This expression is the path to the first data file

In [6]:
DATA_DIR / data_files[0]

PosixPath('/Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/xlsx/210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx')

#### Experiment #1

With no other arguments we expect the Pandas method that reads XLS files to return the first sheet in the spreadsheet.

```python
> df = pd.read_excel(DATA_DIR / data_files[0])
```

That fails with `ValueError: Worksheet index 0 is invalid, 0 worksheets found`

#### Experiment #2

It looks like we have older "macro-enabled" (`.xlsm`) files.  The suggested fix is to use a different parsing engine.

In [7]:
df = pd.read_excel(DATA_DIR / data_files[0], engine='calamine')

That worked.  We got the first ("Overall") sheet that lists the measurements included in the file.

There is an issue, though -- the first row has the sheet name, and the column names are in the second row.  Pass another paramter to tell Pandas to get the column names from the second row.

In [8]:
df = pd.read_excel(DATA_DIR / data_files[0], engine='calamine', header=1)

In [9]:
df[0:4]

,Variable,Value,Unit,Channel,Image,Name,Surpass Object,Time,ID
0,Data Intensity Max,46125.000000,NaN,1.0,Image 1,NaN,Volume,1.0,NaN
1,Data Intensity Max,42515.000000,NaN,2.0,Image 1,NaN,Volume,1.0,NaN
2,Data Intensity Max,15856.000000,NaN,3.0,Image 1,NaN,Volume,1.0,NaN
3,Data Intensity Mean,816.711975,NaN,1.0,Image 1,NaN,Volume,1.0,NaN


Get the frame with max intensity data (channel 1)

In [10]:
imax_1 = pd.read_excel(DATA_DIR / data_files[0], engine='calamine', header=1, sheet_name='Intensity Max Ch=1 Img=1')

In [11]:
len(imax_1)

4579

In [12]:
imax_1.head()

,Intensity Max,Unit,Category,Channel,Image,Surpass Object,Time,ID
0,6754,NaN,Surface,1,Image 1,prg1_tz,1,0
1,4964,NaN,Surface,1,Image 1,prg1_ep,1,0
2,6293,NaN,Surface,1,Image 1,prg1_mp,1,0
3,9391,NaN,Surface,1,Image 1,prg1_lp,1,0
4,6072,NaN,Surface,1,Image 1,prg1_dk,1,0


#### `ExcelFile` Object

Since we're reading multiple sheets we want to use the `ExcelFile` class.  It opens the file and loads it into memory, and then we can call a file method to parse each sheet.

This statement opens the file and saves it so we can examine its properties.  One useful property is a list of sheet names.

In [13]:
f = pd.ExcelFile(DATA_DIR / data_files[0], engine='calamine')

In [14]:
len(f.sheet_names)

39

Hmmm, that's not good, it's truncated at 25 sheets.  Why?

In [15]:
f.sheet_names[0:5]

['Overall',
 'Intensity Max Ch=1 Img=1',
 'Intensity Max Ch=2 Img=1',
 'Intensity Max Ch=3 Img=1',
 'Intensity Mean Ch=1 Img=1']

### Output File

In [16]:
OUTPUT_DIR

PosixPath('/Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/Demo')

In [17]:
! tree /Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/Demo

/Users/conery/Research/Projects/LibudaLab/GAP/PRG-1/Demo
├── ~$Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx
├── Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx
├── Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g07_stitchedtif.xlsx
└── Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g09_stitched.xlsx

1 directory, 4 files


So we have one output for each input, where the names has "Linearized" prepended to the input file name.

In [18]:
output_files = sorted(os.listdir(OUTPUT_DIR))

In [19]:
output_files

['Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx',
 'Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g07_stitchedtif.xlsx',
 'Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g09_stitched.xlsx',
 '~$Linearized_210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx']

In [20]:
ddf = pd.read_excel(OUTPUT_DIR/output_files[0], engine='calamine')

In [21]:
len(ddf)

4583

In [22]:
ddf.head()

,Unit,Category,Collection,Name,Surpass Object,Time,ID,GonadID,X,Y,...,displacement,distance1,distance2,delta_distance1,delta_distance2,segment,tX,ntX,meiosis_stage,ntX_pachytene_normd
0,µm,prg1_dk,Position,NaN,prg1_dk,1,0,NaN,21.794001,138.720001,...,16.818130,2.058651,18.945127,-0.901987,-0.098013,1,2.058651,0.008604,PMT,0.008604
1,µm,prg1_dk,Position,NaN,prg1_dk,1,2,NaN,26.146000,135.149994,...,12.633132,5.823043,15.180735,-0.722762,-0.277238,1,5.823043,0.024337,PMT,0.024337
2,µm,prg1_dk,Position,NaN,prg1_dk,1,4,NaN,25.013000,137.296005,...,13.667280,3.627686,17.376091,-0.827284,-0.172716,1,3.627686,0.015162,PMT,0.015162
3,µm,prg1_dk,Position,NaN,prg1_dk,1,6,NaN,24.264999,138.662994,...,14.352286,2.228068,18.775710,-0.893921,-0.106079,1,2.228068,0.009312,PMT,0.009312
4,µm,prg1_dk,Position,NaN,prg1_dk,1,7,NaN,26.162001,137.666000,...,12.502630,3.310372,17.693406,-0.842392,-0.157608,1,3.310372,0.013836,PMT,0.013836


### Measurements

The coordinates of the points that mark the center of a germline are in the "Position" sheet.

In [23]:
data_files[0]

'210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx'

In [24]:
mf = pd.read_excel(DATA_DIR / data_files[0], engine='calamine', header=1, sheet_name="Position")

In [25]:
mf.head()

,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
0,21.794001,138.720001,7.700,µm,Surface,Position,NaN,prg1_dk,1,0
1,19.615000,141.886993,5.377,µm,Surface,Position,NaN,prg1_dk,1,1
2,26.146000,135.149994,7.900,µm,Surface,Position,NaN,prg1_dk,1,2
3,21.146999,141.274994,7.451,µm,Surface,Position,NaN,prg1_dk,1,3
4,25.013000,137.296005,7.400,µm,Surface,Position,NaN,prg1_dk,1,4


The Category column has "Surface" for a data point or "MeasurementPoint" for a end-point of a line segment.

In [26]:
sf = mf[mf.Category == "MeasurementPoint"]

In [27]:
sf.head()

,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
4579,38.500999,141.542007,0.521,µm,MeasurementPoint,Position,A,Measurement Points 1,1,0
4580,39.457001,120.559998,6.300,µm,MeasurementPoint,Position,B,Measurement Points 1,1,1
4581,40.560001,110.825996,4.700,µm,MeasurementPoint,Position,C,Measurement Points 1,1,2
4582,43.907001,99.404999,4.300,µm,MeasurementPoint,Position,D,Measurement Points 1,1,3
4583,48.821999,81.403999,4.900,µm,MeasurementPoint,Position,E,Measurement Points 1,1,4


In [28]:
sf[['Name','Position X','Position Y']]

,Name,Position X,Position Y
4579,A,38.500999,141.542007
4580,B,39.457001,120.559998
4581,C,40.560001,110.825996
4582,D,43.907001,99.404999
4583,E,48.821999,81.403999
4584,F,54.426998,66.403999
4585,G,61.472000,53.318001
4586,H,67.041000,44.591000
4587,I,72.586998,25.798000
4588,J,69.010002,14.931000


Save the measurement names and locations in a CSV file

In [29]:
sf[['Position X','Position Y','Name']].to_csv("measurements.csv", index=False)

Save a random sample of point locations to use in tests

In [30]:
mf[mf.Category=='Surface'].sample(20).to_csv("points.csv", index=False)

### Nucleus IDs

D'oh, first version overlooked the fact that the "ID" column is not a unique ID.  We need to concatenate the "Surpass Object" column (strings of the form `prg1_dk`, `prg1_ep`, ...) with the number in the ID column.

In [31]:
pf = pd.read_excel(DATA_DIR / data_files[0], engine='calamine', header=1, sheet_name='Position')

In [32]:
pf.head()

,Position X,Position Y,Position Z,Unit,Category,Collection,Name,Surpass Object,Time,ID
0,21.794001,138.720001,7.700,µm,Surface,Position,NaN,prg1_dk,1,0
1,19.615000,141.886993,5.377,µm,Surface,Position,NaN,prg1_dk,1,1
2,26.146000,135.149994,7.900,µm,Surface,Position,NaN,prg1_dk,1,2
3,21.146999,141.274994,7.451,µm,Surface,Position,NaN,prg1_dk,1,3
4,25.013000,137.296005,7.400,µm,Surface,Position,NaN,prg1_dk,1,4


In [37]:
pf['Surpass Object'].head() + pf['ID'].apply(str)

0       prg1_dk0
1       prg1_dk1
2       prg1_dk2
3       prg1_dk3
4       prg1_dk4
          ...   
4591         NaN
4592         NaN
4593         NaN
4594         NaN
4595         NaN
Length: 4596, dtype: str

In [38]:
pd.Series(pf['Surpass Object'].head() + pf['ID'].apply(str), name='ID')

0       prg1_dk0
1       prg1_dk1
2       prg1_dk2
3       prg1_dk3
4       prg1_dk4
          ...   
4591         NaN
4592         NaN
4593         NaN
4594         NaN
4595         NaN
Name: ID, Length: 4596, dtype: str